In [1]:
import os
os.chdir('../../../')

In [9]:
from amed.networks import AMED_predictor

predictor = AMED_predictor(sampler_stu='amed', sampler_tea='euler', bottleneck_input_dim=32*32)

In [10]:
import torch

def get_amed_prediction(AMED_predictor, t_cur, t_next, unet_out):
    unet_enc = torch.mean(unet_out, dim=1)
    output = AMED_predictor(unet_enc, t_cur, t_next)
    output_list = [*output]
    
    if len(output_list) == 2:
        try:
            use_scale_time = AMED_predictor.module.scale_time
        except:
            use_scale_time = AMED_predictor.scale_time
        if use_scale_time:
            r, scale_time = output_list
            r = r.reshape(-1, 1, 1, 1)
            scale_time = scale_time.reshape(-1, 1, 1, 1)
            scale_dir = torch.ones_like(scale_time)
        else:
            r, scale_dir = output_list
            r = r.reshape(-1, 1, 1, 1)
            scale_dir = scale_dir.reshape(-1, 1, 1, 1)
            scale_time = torch.ones_like(scale_dir)
    elif len(output_list) == 3:
        r, scale_dir, scale_time = output_list
        r = r.reshape(-1, 1, 1, 1)
        scale_dir = scale_dir.reshape(-1, 1, 1, 1)
        scale_time = scale_time.reshape(-1, 1, 1, 1)
    else:
        r = output.reshape(-1, 1, 1, 1)
        scale_dir = torch.ones_like(r)
        scale_time = torch.ones_like(r)
    return r, scale_dir, scale_time

In [13]:
t_cur = torch.randn(1,)
t_next = torch.randn(1,)
unet_out = torch.randn(10, 4, 32, 32)
r, scale_dir, scale_time = get_amed_prediction(predictor, t_cur, t_next, unet_out)
r.shape

torch.Size([10, 1, 1, 1])